In [63]:
import pandas as pd

# -------------- Load Dataset --------------
cic = pd.read_csv("../CICMal2017 Dataset - Extracted Features with specified APK Benign vs Malicious.csv")
cic['Dataset'] = "CICAndMal2017"

drebin = pd.read_csv("../DREBIN Dataset - APK signatures.csv", low_memory=False)
drebin['Dataset'] = "Drebin"

amsf = pd.read_csv("../AMSF Dataset - Extracted Features with Malware signatures (and Benign vs Malicious).csv")
amsf['Dataset'] = "AMSF"


am = pd.read_csv("../AM.csv", sep=";")
am['Dataset'] = "AM"


In [64]:
# -------------- Data-preprocessing [Benign = 0, Malicious = 1] --------------

#CICMal2017
cic_cols = ["App", "Package", "Category", "Description", 
             "Rating", "Number of ratings", "Price", "Related apps"]

cic = cic.drop(columns=[c for c in cic_cols if c in cic.columns])
cic.rename(columns={"Class": "class"}, inplace=True)

cic['class'] = cic['class'].replace({
    'B': 0, 'benign': 0, 'benignware': 0,
    'S': 1, 'malicious': 1, 'malware': 1
})
cic = cic.fillna(0)


#Drebin
drebin['class'] = drebin['class'].replace({
    'B': 0, 'benign': 0, 'benignware': 0,
    'S': 1, 'malicious': 1, 'malware': 1
})
drebin = drebin.fillna(0)

for col in drebin.columns:
    if col != "class" and col != "Dataset":
        drebin[col] = drebin[col].astype(str)
      
        
#AMSF
amsf['class'] = amsf['class'].replace({
    'B': 0, 'benign': 0, 'benignware': 0,
    'S': 1, 'malicious': 1, 'malware': 1
})
amsf = amsf.fillna(0)


#AM
am_cols = [
    "name", ".//MD5", "version", ".//Min_SDK", ".//Min_Screen", ".//Min_OpenGL", 
    ".//Supported_CPU", ".//Signature", ".//Developer", ".//Organization", 
    ".//Locality", ".//Country", ".//State", "description", "rating_number", 
    "rating_count"
]
am = am.drop(columns=[c for c in am_cols if c in am.columns])

am = am.rename(columns={"LABEL": "class"})

am['class'] = am['class'].replace({
    'B': 0, 'benign': 0, 'benignware': 0,
    'S': 1, 'malicious': 1, 'malware': 1
})
am = am.fillna(0)


all_columns = set(cic.columns) | set(drebin.columns) | set(amsf.columns) | set(am.columns)

cic = cic.reindex(columns=all_columns, fill_value=0)
drebin = drebin.reindex(columns=all_columns, fill_value=0)
amsf = amsf.reindex(columns=all_columns, fill_value=0)
am = am.reindex(columns=all_columns, fill_value=0)

for df in [cic, drebin, amsf, am]:
    non_numeric_cols = df.select_dtypes(exclude=['int64', 'float64']).columns
    non_numeric_cols = [col for col in non_numeric_cols if col not in ['class', 'Dataset']]
    df[non_numeric_cols] = 0

am

C:\Users\raf_s\AppData\Local\Temp\ipykernel_10584\3219068450.py:18: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  drebin['class'] = drebin['class'].replace({
C:\Users\raf_s\AppData\Local\Temp\ipykernel_10584\3219068450.py:48: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  am['class'] = am['class'].replace({


,android.content.Context.getPackageName,java.util.Locale.getCountry,android.permission.READ_FRAME_BUFFER,android.intent.action.ACTION_SHUTDOWN,Runtime.exec,Services that cost you money : send SMS messages (D),java.lang.String.length,android.permission.SIGNAL_PERSISTENT_PROCESSES,System tools : force stop other applications (S),Your accounts : YouTube usernames (D),...,android.widget.LinearLayout.setBackgroundColor,android.net.wifi.WifiManager,java.lang.StringBuilder.toString,android.graphics.drawable.GradientDrawable,android.graphics.Paint.setColor,java.lang.Math.max,java.lang.Math.random,java.net.HttpURLConnection.setDoOutput,android.graphics.Typeface,android.content.pm.Signature
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11471,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
11472,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
11473,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
11474,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [65]:
# -------------- Data Aggregation --------------
aggregate = pd.concat([cic, drebin, amsf, am], axis=0, join="outer", ignore_index=True)

cols = [c for c in aggregate.columns if c not in ['class', 'Dataset']]
aggregated = aggregate[['class'] + cols + ['Dataset']]

missing_values = aggregated.isnull().sum().sum()
non_numeric = aggregated.select_dtypes(exclude=['int64', 'float64']).columns
print(f"Total missing values in aggregated dataset: {missing_values}")
print("Aggregated shape:", aggregated.shape)
print("Non-numeric columns in AM:", non_numeric.tolist())
aggregated.sample(5, random_state=42)

Total missing values in aggregated dataset: 0
Aggregated shape: (61530, 1080)
Non-numeric columns in AM: ['Dataset']


,class,android.content.Context.getPackageName,java.util.Locale.getCountry,android.permission.READ_FRAME_BUFFER,android.intent.action.ACTION_SHUTDOWN,Runtime.exec,Services that cost you money : send SMS messages (D),java.lang.String.length,android.permission.SIGNAL_PERSISTENT_PROCESSES,System tools : force stop other applications (S),...,android.net.wifi.WifiManager,java.lang.StringBuilder.toString,android.graphics.drawable.GradientDrawable,android.graphics.Paint.setColor,java.lang.Math.max,java.lang.Math.random,java.net.HttpURLConnection.setDoOutput,android.graphics.Typeface,android.content.pm.Signature,Dataset
11316,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,CICAndMal2017
25630,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,CICAndMal2017
5954,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,CICAndMal2017
49090,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,AMSF
54803,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,AM


In [66]:
aggregated.to_csv("Aggregated_MALDROID_Datasets.csv", index=False)